# 🔢 06 - Sorting and Filtering in Pandas

Notebook 03 introduced boolean masks and `.loc[]`. This notebook puts them to work: sorting, building long filters you can still read, `.query()`, searching text, and turning a question into a sorted report.

## ✅ Learning Goals
- Sort rows with `.sort_values()`, including by multiple columns, and know that sorting returns a **new** DataFrame
- Filter data with one or more conditions, using `&`, `|`, and `~`
- Keep long filters readable with **named masks**
- Use `.query()`, including `in`, `@variables`, and column names with spaces
- Search text columns with `.str.contains()` (including patterns like `'son|ez'`), `.str.startswith()`, and `.str.endswith()`
- Answer a question with a filtered, sorted report

> **The core habit:** filter with clear conditions, sort only after selecting the rows you need, and use `.query()` or named masks when an expression gets long.

## 1. Load the Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('students.csv')
df.head()

## 2. Sorting Data

In [ ]:
# Sort by math_score, ascending (the default)
df.sort_values(by='math_score')

In [ ]:
# Sort by science_score, descending
df.sort_values(by='science_score', ascending=False)

In [ ]:
# Sort by multiple columns — science_score descending, then math_score ascending
# to break ties
df.sort_values(by=['science_score', 'math_score'], ascending=[False, True])

The **order** of the `by=` list sets the priority: sort by the first column, and use the second only to break **ties** in the first. `ascending=` takes a matching list, one direction per column.

`.sort_values()` returns a **new**, sorted DataFrame. The original keeps its order unless you assign the result:

In [ ]:
by_math = df.sort_values('math_score')            # lowest math score first
print('sorted copy starts with:', by_math['first_name'].iloc[0])
print('df still starts with:   ', df['first_name'].iloc[0])

💡 For "the top 5" or "the bottom 3" of one column, `df.nlargest(5, 'math_score')` and `df.nsmallest(3, 'math_score')` are shortcuts (notebook 03).

## 3. Filtering with Conditions

In [ ]:
# Who scored at least 90 in math?
df[df['math_score'] >= 90]

In [ ]:
# Who scored less than 70 in either subject?
df[(df['math_score'] < 70) | (df['science_score'] < 70)]

## 4. Combining Filters
Recap from notebook 03: `&` means **and**, `|` means **or**, `~` means **not**, and every condition goes in its own parentheses. (Python's `and`/`or` keywords raise a `ValueError` here.)

In [ ]:
# Students with 87+ in both subjects
df[(df['math_score'] >= 87) & (df['science_score'] >= 87)]

### Named Masks for Long Filters
A filter with three or four conditions turns into one long, hard-to-read line. Instead, give each condition a **name** first, then combine the names. It reads like a sentence, and you can check each mask on its own:

In [ ]:
strong_math = df['math_score'] >= 85
strong_science = df['science_score'] >= 85
late_alphabet = df['last_name'] >= 'M'           # text compares alphabetically

print('strong in math:', strong_math.sum())      # check each piece separately
df[strong_math & strong_science & late_alphabet]

`.isin()`, `.between()`, and `~` (notebook 03) all combine the same way. For example, `~df['last_name'].isin(['Smith', 'Jones'])` is "everyone **except** Smith and Jones".

## 5. Using `.query()` for Readability

In [ ]:
# The same filter, written with .query()
df.query('math_score >= 87 and science_score >= 87')

Inside `.query()`, you write `and`, `or`, and `not` (not `&`, `|`, `~`), and you can test membership with `in`:

In [ ]:
df.query("last_name in ['Wood', 'Wong', 'Clark']")

### Using Python Variables with `@`
Put `@` in front of a **Python variable** to use its value. Without the `@`, pandas looks for a **column** with that name:

In [ ]:
minimum = 90
df.query('math_score >= @minimum and science_score >= @minimum')

### Column Names with Spaces
`.query()` can't read a name like `Unit Price` directly. Wrap it in **backticks**. Better still, standardize the names first (notebook 05), and the problem goes away:

In [ ]:
sales = pd.DataFrame({'Product Name': ['Notebook', 'Pen', 'Lamp'], 'Unit Price': [4.99, 1.50, 24.99]})
print(sales.query('`Unit Price` < 5'))

sales.columns = sales.columns.str.strip().str.lower().str.replace(' ', '_')
sales.query('unit_price < 5')

**Masks or `.query()`?** Both are good. Use whichever reads more clearly to the next person. `.query()` shines for long numeric conditions, and masks are better for text methods like `.str.contains()`.

## 6. Searching Text with `.str`
Remember the string methods from Module 02? A text column's `.str` gives you those same methods, applied to **every** value at once, and the result works as a filter mask:

In [ ]:
df[df['last_name'].str.startswith('W')]

In [ ]:
df[df['last_name'].str.endswith('son')]

In [ ]:
# 'an' anywhere in the name, ignoring capitalization
df[df['first_name'].str.contains('an', case=False, na=False)]

- `case=False` matches `Angel` and `Danielle` alike, whatever the capitalization.
- `na=False` treats missing names as "no match". Without it, a single `NaN` in the column makes the filter fail.

Other string methods work the same way: `.str.upper()`, `.str.strip()`, `.str.len()`, `.str.replace()`, and more.

### Patterns: `|` Means "Or" Inside `.str.contains()`
`.str.contains()` treats its search text as a **regular expression** (a pattern), where `|` means "or":

In [ ]:
df[df['last_name'].str.contains('son|ez', na=False)]

That's powerful, but it means some characters, like `+`, `.`, `(`, and `|`, have **special** meanings. Searching for a literal `C++` quietly matches any `C`. Turn patterns off with `regex=False`:

In [ ]:
books = pd.Series(['C++ Primer', 'C Programming', 'Python Basics'])
print('as a pattern:', books.str.contains('C++').tolist())                # 'C Programming' matches too!
print('literal text:', books.str.contains('C++', regex=False).tolist())

### Case-Insensitive Starts and Ends
`.str.startswith()` and `.str.endswith()` have no `case=` option. Lowercase the column first, then compare with lowercase text:

In [ ]:
df[df['first_name'].str.lower().str.startswith('a')]

### Combining Text Searches with Other Conditions
A text mask combines with number masks like any other mask, and named masks keep it readable:

In [ ]:
ends_in_y = df['first_name'].str.lower().str.endswith('y')
passed_math = df['math_score'] >= 70

df[ends_in_y & passed_math]

## 7. Filter, Then Sort: Building a Report
Most real questions combine everything: **filter** the rows, pick the **columns**, then **sort**. Filter first, so you only sort the rows you actually need:

In [ ]:
df.loc[df['science_score'] >= 90, ['first_name', 'last_name', 'science_score']].sort_values(
    'science_score', ascending=False)

### A Complete Walkthrough
**Question:** *Which students scored at least 80 in **both** subjects and have a last name that ends in "son" or "ez", or starts with "W"? Rank them by their average.*

Break the question into named masks, combine them, then build the report:

In [ ]:
both_80 = (df['math_score'] >= 80) & (df['science_score'] >= 80)
name_match = df['last_name'].str.contains('son|ez', na=False) | df['last_name'].str.startswith('W', na=False)

report = df[both_80 & name_match].copy()
report['average'] = (report['math_score'] + report['science_score']) / 2
report = report[['first_name', 'last_name', 'math_score', 'science_score', 'average']].sort_values(
    'average', ascending=False)
report

`.copy()` makes it clear that `report` is a separate table, so adding the `average` column doesn't touch `df` (notebook 03). The everyday pattern is:

```python
df.loc[row_condition, ['column_a', 'column_b']].sort_values('column_a')
```

## 🏋️ Practice

### Practice 1 — Alphabetical Sort (easy)
Sort the dataset by `first_name`, alphabetically.

In [ ]:
# Your code here


### Practice 2 — Close Scores (medium)
Find students whose `math_score` and `science_score` are within **2 points** of each other. Hint: `(df['math_score'] - df['science_score']).abs() <= 2` makes the mask. Sort the result by `last_name`.

In [ ]:
# Your code here


### Practice 3 — Query with OR (harder)
Use `.query()` to find students who scored over 95 in *either* subject (hint: `.query()` uses `or`, not `|`).

In [ ]:
# Your code here


### Practice 4 — Search a Bigger Dataset (medium)
Load `pokemon.csv`. Find every Pokémon whose `Name` contains `'mega'`, regardless of capitalization, and sort them by `Total`, highest first. How many are there?

In [ ]:
poke = pd.read_csv('pokemon.csv')
# Your code here


### Practice 5 — Query with Variables (medium)
Create two variables, `min_math` and `min_science`. Use `.query()` with `@` to find students who meet **both** minimums, sorted by `math_score` from highest to lowest. Then change the variables and re-run, without touching the query itself.

In [ ]:
# Your code here


## ✏️ Your Turn
1. Sort the dataset by `last_name`, then by `first_name` to break ties.
2. Use named masks to find students with a `math_score` from 70 to 85 (`.between()`) whose `first_name` contains `'e'` (any capitalization).
3. Use `.query()` to select rows matching a condition of your own choosing.
4. Add an `average_score` column, sort by it descending, and show just the top 5 students' names and averages.

In [ ]:
# Your turn here


## 🔥 Challenge (Optional) — A Product Report
Using the product table below, answer: *Which Office and Travel products cost less than \$30, have at least 10 in stock, and have `'note'` or `'pen'` in their name (any capitalization)?*

1. Build a **named mask** for each condition, and combine them.
2. Show only `product_name`, `department`, `price`, `stock`, and `rating`, sorted by `department` A–Z, then `price` from highest to lowest.
3. Answer the same question again with `.query()` for the number conditions, plus a mask for the name. Which version is easier to read?

In [ ]:
products = pd.DataFrame({
    'product_id':   [101, 102, 103, 104, 105, 106],
    'product_name': ['Notebook', 'Travel Backpack', 'Blue Pen', 'Desk Lamp', 'Water Bottle', 'Notebook Stand'],
    'department':   ['Office', 'Travel', 'Office', 'Home', 'Travel', 'Office'],
    'price':        [4.99, 39.99, 1.50, 24.99, 12.50, 29.99],
    'stock':        [32, 14, 120, 8, 25, 11],
    'rating':       [4.6, 4.8, 4.2, 4.5, 4.7, 4.3],
})
# Your code here


---
### 📝 Summary
| Task              | Code Example                                 |
|-------------------|------------------------------------------------|
| Sort by column     | `df.sort_values(by='col')` (returns a new DataFrame) |
| Sort by several    | `df.sort_values(by=['a', 'b'], ascending=[True, False])` |
| Filter condition   | `df[df['col'] > value]`                       |
| Combine filters    | `df[(cond1) & (cond2)]` (and), or, `~` (not)   |
| Named masks        | `is_x = ...; is_y = ...; df[is_x & is_y]`      |
| Query method       | `df.query('col > value and col2 < value')`    |
| Query with variables | `df.query('col >= @minimum')`               |
| Query with spaces  | ``df.query('`Unit Price` < 5')``              |
| Search text        | `df[df['col'].str.contains('x', case=False, na=False)]` |
| Pattern / literal  | `.str.contains(pattern)`, add `regex=False` for literal text |
| Filter, then sort  | `df.loc[cond, cols].sort_values('col')`       |